In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/california-homelessness-prediction-challenge/sample_submission.csv
/kaggle/input/california-homelessness-prediction-challenge/train.csv
/kaggle/input/california-homelessness-prediction-challenge/test.csv
/kaggle/input/california-homelessness-prediction-challenge/sandbox_submission.csv


In [2]:
import pandas as pd

def analyze_homelessness_correlation(file_path='/kaggle/input/california-homelessness-prediction-challenge/train.csv'):
    """
    Loads the homelessness dataset, calculates the correlation of all features
    with the HOMELESS_RATE, and prints the sorted results.

    """
    try:
        # Load the uploaded CSV file
        df = pd.read_csv(file_path)

        # Ensure HOMELESS_RATE is in the dataframe
        if 'HOMELESS_RATE' not in df.columns:
            print("Error: 'HOMELESS_RATE' column not found in the file.")
            return

        # Calculate the correlation matrix
        correlation_matrix = df.corr(numeric_only=True)

        # Get the correlations with the target variable 'HOMELESS_RATE'
        homeless_rate_correlation = correlation_matrix['HOMELESS_RATE'].sort_values(ascending=False)

        # Print the results
        print("--- Correlation with HOMELESS_RATE ---")
        print(homeless_rate_correlation)

    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Execute the analysis
analyze_homelessness_correlation()

--- Correlation with HOMELESS_RATE ---
HOMELESS_RATE                          1.000000
AGE_25_34_PCT                          0.442950
NONVETERAN_POP_PCT                     0.356066
AGE_35_44_PCT                          0.288187
RACE_BLACK_NH_PCT                      0.222333
DISABILITY_POP_PCT                     0.158287
AGE_18_24_PCT                          0.076610
RACE_ASIAN_NH_PCT                      0.067259
NONFAMILY_SINGLE_FEMALE_PCT            0.065772
INDIVIDUALS_NOT_IN_FAMILY_UNITS_PCT    0.065772
RACE_HISPANIC_ANY_PCT                  0.049673
RACE_PACIFIC_NH_PCT                    0.030108
MULTI_PERSON_NONFAMILY_HH_PCT          0.021117
RACE_NATIVE_NH_PCT                     0.007965
RACE_TWO_OR_MORE_NH_PCT               -0.038063
NODISABILITY_POP_PCT                  -0.060266
AGE_65_69_PCT                         -0.066369
AGE_45_54_PCT                         -0.111185
AGE_80_PLUS_PCT                       -0.112870
AGE_62_64_PCT                         -0.125595
A

## Conclusion and Social Significance
The correlation analysis reveals critical insights into the socio-economic factors associated with homelessness in California.

## 📈 Key Positive Correlations: 
The data indicates a notable positive correlation between the homelessness rate and the percentage of the population aged 25-44 (AGE_25_34_PCT, AGE_35_44_PCT) and the Black, Non-Hispanic community (RACE_BLACK_NH_PCT).

## 📉 Key Negative Correlations: 
Conversely, a strong negative correlation exists with factors related to family stability, such as the total number of family households (FAMILY_HH_TOTAL), especially those with children (FAMILY_HH_CHILD_LT18_PCT), and the proportion of the population under 18 (AGE_U18_PCT).

## Social Significance
This data-driven insight has profound social significance for policymakers and community organizations:

Shift from Individual Blame to Systemic Issues: The correlations suggest that homelessness is not merely a result of individual choices but is deeply intertwined with broader demographic and economic pressures. The vulnerability of the prime working-age population (25-44) points towards issues like wage stagnation, the high cost of living, and a lack of affordable housing rather than an unwillingness to work.

## Highlights Structural Inequality: 
The significant correlation with the Black community underscores the impact of systemic racism and historical disadvantages that lead to disparities in housing, employment, and economic stability. Addressing homelessness therefore requires a direct focus on racial equity.

## Identifies Protective Factors: 
The strong negative correlation with family households suggests that stable family structures and community support networks act as a crucial protective barrier against homelessness. This indicates that policies aimed at supporting families, such as childcare assistance and affordable family housing, could be effective preventative measures.

In conclusion, this analysis allows for a more nuanced understanding of homelessness. It encourages a shift in focus from emergency response to proactive, targeted interventions. By identifying the specific demographic groups at highest risk and the social structures that offer protection, public resources can be allocated more effectively to prevent homelessness before it begins.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def train_and_evaluate_model(file_path='/kaggle/input/california-homelessness-prediction-challenge/train.csv'):
    """
    Loads data, trains a Gradient Boosting Regressor model,
    and evaluates its performance using multiple metrics.

    """
    try:
        # 1. Load Data
        df = pd.read_csv(file_path)

        # 2. Define Features (X) and Target (y)
        # Drop the target variable and the non-numeric ID
        X = df.drop(['HOMELESS_RATE', 'ID'], axis=1)
        y = df['HOMELESS_RATE']

        # 3. Split Data into Training and Testing sets
        # 80% for training, 20% for testing
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # 4. Initialize and Train the Model
        # GradientBoostingRegressor is a powerful model for tabular data
        print("Training the Gradient Boosting model...")
        model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
        model.fit(X_train, y_train)
        print("Training complete.")

        # 5. Make Predictions on the Test Set
        y_pred = model.predict(X_test)

        # 6. Evaluate the Model
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        print("\n--- Model Evaluation Results ---")
        print(f"RMSE (Root Mean Squared Error): {rmse:.6f}")
        print(f"MAE (Mean Absolute Error):     {mae:.6f}")
        print(f"R² Score (Coefficient of Determination): {r2:.6f}")
        print("---------------------------------")

    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Execute the training and evaluation
train_and_evaluate_model()

Training the Gradient Boosting model...
Training complete.

--- Model Evaluation Results ---
RMSE (Root Mean Squared Error): 0.014760
MAE (Mean Absolute Error):     0.006716
R² Score (Coefficient of Determination): -26.614827
---------------------------------


## What can be done with the prediction results (speculation)

The prediction results from this model hold value beyond mere numbers and can lead to concrete actions addressing homelessness.

1. Identifying areas requiring intervention (Targeted Intervention) 🗺️
Areas (at the census tract level) predicted by the model to have high future homelessness rates can be identified as “hotspots.” This enables governments and NPOs to concentrate resources in areas most in need, even with limited resources.

Example: Prioritize deploying social worker patrols, mobile free meal services, and mental health counseling centers in the top 5% of areas with the highest predicted values.

2. Enabling Preventive Action 🛡️
Even if problems are not yet apparent, a “preventive approach” becomes possible for areas predicted to face increased future risk, allowing action before issues arise.

Example: If the model learns that rising unemployment correlates with increased homelessness, implement rent assistance programs and job re-entry seminars early in at-risk areas where unemployment begins to rise.

3. Policy Effect Simulation 🧪
Using trained models, we can simulate how policy changes will impact future homelessness rates.

Example: Calculate “How much would the predicted homelessness rate decrease if low-income housing were increased by 10% in a given area (by changing the value of relevant features)?” to select the most effective policy option based on data.

4. Budgeting & Accountability 📊
Data-driven, objective forecasts provide strong justification for government agencies when requesting budgets from the legislature for new support programs. They also serve as objective evidence when explaining to citizens why tax funds should be allocated to specific areas.
Thus, predictive models are not merely tools for knowing the future; they act as a compass, guiding us on where, when, and what interventions to make to build a better future.

In [4]:
import pandas as pd

def analyze_high_homeless_rate_areas(file_path='/kaggle/input/california-homelessness-prediction-challenge/train.csv'):
    """
    Identifies areas with the highest homeless rates and analyzes their
    common characteristics compared to the overall average.

    """
    try:
        # Load the data
        df = pd.read_csv(file_path)

        # Define what constitutes a "high" homeless rate.
        # Let's use the top 10% of areas as our sample group.
        quantile_threshold = df['HOMELESS_RATE'].quantile(0.90)
        high_rate_df = df[df['HOMELESS_RATE'] >= quantile_threshold]

        if high_rate_df.empty:
            print("No areas found in the top 10% quantile. The threshold might be too high or the data is uniform.")
            return

        # Drop non-feature columns for analysis
        features_df = df.drop(columns=['ID', 'HOMELESS_RATE'])
        high_rate_features_df = high_rate_df.drop(columns=['ID', 'HOMELESS_RATE'])

        # Calculate the average characteristics for the high-rate group and the overall dataset
        high_rate_avg = high_rate_features_df.mean()
        overall_avg = features_df.mean()

        # Create a comparison dataframe to see the differences clearly
        comparison_df = pd.DataFrame({
            'High-Rate Areas Average': high_rate_avg,
            'Overall Average': overall_avg
        })
        comparison_df['Difference'] = comparison_df['High-Rate Areas Average'] - comparison_df['Overall Average']
        comparison_df['Ratio (High/Overall)'] = comparison_df['High-Rate Areas Average'] / comparison_df['Overall Average']

        # Sort by the ratio to see the most pronounced differences
        sorted_comparison = comparison_df.sort_values(by='Ratio (High/Overall)', ascending=False)

        print(f"--- Characteristics of Areas with the Highest Homelessness Rates (Top 10%) ---")
        print(f"\nThese areas, when compared to the average, have notably different demographics.")
        print("Below are the most significant differences, sorted by how much higher the feature is compared to the average:\n")
        print(sorted_comparison)

    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Run the analysis
analyze_high_homeless_rate_areas()

--- Characteristics of Areas with the Highest Homelessness Rates (Top 10%) ---

These areas, when compared to the average, have notably different demographics.
Below are the most significant differences, sorted by how much higher the feature is compared to the average:

                                     High-Rate Areas Average  Overall Average  \
RACE_BLACK_NH_PCT                                   0.084946         0.044100   
RACE_PACIFIC_NH_PCT                                 0.004461         0.003549   
INDIVIDUALS_NOT_IN_FAMILY_UNITS_PCT                 0.071258         0.058919   
NONFAMILY_SINGLE_FEMALE_PCT                         0.071258         0.058919   
AGE_25_34_PCT                                       0.172782         0.148382   
MULTI_PERSON_NONFAMILY_HH_PCT                       0.021613         0.018738   
RACE_HISPANIC_ANY_PCT                               0.375366         0.328907   
AGE_18_24_PCT                                       0.095235         0.083695   


There does not appear to be a correlation with the regions (IDs) themselves that have high HOMELESS_RATE. However, it can be understood that 

## “regions (IDs) with high HOMELESS_RATE share common socioeconomic characteristics.”

We extracted the top 10% of regions with the highest homelessness rates and compared the average values of their characteristics to the overall data average.

Analysis revealed that regions with the highest homelessness rates exhibit the following distinct characteristics:

## Characteristics of Areas with High Homeless Rates
📈 Features with Particularly High Proportions
Percentage of Non-Hispanic Black Residents (RACE_BLACK_NH_PCT):

This showed the most significant difference, approximately 1.93 times the overall average. This strongly suggests the homelessness issue disproportionately impacts specific communities.

## Percentage of individuals not in family units (INDIVIDUALS_NOT_IN_FAMILY_UNITS_PCT):

The proportion of people living alone without family ties (especially single women) was approximately 1.21 times higher than the overall average. This indicates that social isolation without supportive family networks is a risk factor.

## Percentage of Young Adults (AGE_25_34_PCT):

The proportion of the population aged 25 to 34 is approximately 1.16 times higher than the overall average. This indicates that younger generations, who are more prone to economic instability, are likely to fall into homelessness.



### 📉 Characteristics with particularly low rates

## Percentage of family households with children (FAMILY_HH_CHILD_LT18_PCT):

The percentage of family households with children under 18 is only about 0.77 times the overall average. This means that areas with stable family households containing children tend to have lower homelessness rates.

## Percentage of veterans (VETERAN_POP_PCT):

The percentage of veterans is also low, at about 0.77 times the overall average. Support programs for veterans may function as a certain safety net.

## Total number of family households (FAMILY_HH_TOTAL):

The number of households composed of families themselves tends to be lower, at about 0.82 times the overall average.

### Conclusions

These characteristics suggest that areas with high homelessness rates are likely **“regions where the family unit, a key social safety net, is less common, and where economically unstable young people and racial minorities, who are structurally disadvantaged, reside in greater numbers.”**

In other words, **"homelessness is not merely an individual problem but is deeply intertwined with the very structure of the local community."**


In [5]:
features = ['HOMELESS_RATE','RACE_BLACK_NH_PCT', 'INDIVIDUALS_NOT_IN_FAMILY_UNITS_PCT', 'AGE_25_34_PCT','FAMILY_HH_CHILD_LT18_PCT', 'VETERAN_POP_PCT', 'FAMILY_HH_TOTAL'] 

In [6]:
train_df = pd.read_csv('/kaggle/input/california-homelessness-prediction-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/california-homelessness-prediction-challenge/test.csv')
sub = pd.read_csv('/kaggle/input/california-homelessness-prediction-challenge/sample_submission.csv')

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor


def RMSE_GB(df):
    X_train, X_test, y_train, y_test = train_test_split(df.drop('HOMELESS_RATE', axis=1), df['HOMELESS_RATE'], test_size=0.2, random_state=42)
    model = GradientBoostingRegressor()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, y_pred))

def RMSE_LR(df):
    X_train, X_test, y_train, y_test = train_test_split(df.drop('HOMELESS_RATE', axis=1), df['HOMELESS_RATE'], test_size=0.2, random_state=42)
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, y_pred))

def RMSE_svr(df):
    X_train, X_test, y_train, y_test = train_test_split(df.drop('HOMELESS_RATE', axis=1), df['HOMELESS_RATE'], test_size=0.2, random_state=42)
    model = SVR()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, y_pred))

def RMSE_RF(df):
    X_train, X_test, y_train, y_test = train_test_split(df.drop('HOMELESS_RATE', axis=1), df['HOMELESS_RATE'], test_size=0.2, random_state=42)
    model = RandomForestRegressor()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, y_pred))

def RMSE_CatBoost(df):
    X_train, X_test, y_train, y_test = train_test_split(df.drop('HOMELESS_RATE', axis=1), df['HOMELESS_RATE'], test_size=0.2, random_state=42)
    model = CatBoostRegressor(silent=True)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, y_pred))

def RMSE_LGBM(df):
    X_train, X_test, y_train, y_test = train_test_split(df.drop('HOMELESS_RATE', axis=1), df['HOMELESS_RATE'], test_size=0.2, random_state=42)
    model = LGBMRegressor(verbose=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, y_pred))

def RMSE_XGB(df):
    X_train, X_test, y_train, y_test = train_test_split(df.drop('HOMELESS_RATE', axis=1), df['HOMELESS_RATE'], test_size=0.2, random_state=42)
    model = XGBRegressor()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, y_pred))

In [8]:
df = train_df[features]

print('GB ', RMSE_GB(df))
print('LR ', RMSE_LR(df))
print('SVR ',RMSE_svr(df))
print('RF ' ,RMSE_RF(df))
print('CB', RMSE_CatBoost(df))
print('LGB', RMSE_LGBM(df))
print('XGB', RMSE_XGB(df))

GB  0.0074249505374237934
LR  0.003683586623200899
SVR  0.02718690383647199
RF  0.004216262674979871
CB 0.0025064333627343996
LGB 0.0041479142871404095
XGB 0.0027886572359332204


1 : CatBoostRegressor

2 : XgboostRegressor

3 : LinearRegression

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV

from catboost import CatBoostRegressor

X = train_df[features].drop('HOMELESS_RATE', axis = 1)
y = train_df['HOMELESS_RATE']

model_cat = CatBoostRegressor(silent=True)

param_grid = {
    'iterations': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
    'depth': [7, 8, 9, 10],
    'l2_leaf_reg': [1, 3, 5]
}

grid_search = GridSearchCV(estimator=model_cat, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X, y)

best_params = grid_search.best_params_
best_model = grid_search.best_estimator_
print(best_params)
print(best_model)

best_model.fit(X, y)

y_pred = best_model.predict(X)

rmse = np.sqrt(mean_squared_error(y, y_pred))

rmse

{'depth': 8, 'iterations': 100, 'l2_leaf_reg': 1, 'learning_rate': 0.2}


5.326453984617646e-05

In [10]:
test_features =  ['RACE_BLACK_NH_PCT', 'INDIVIDUALS_NOT_IN_FAMILY_UNITS_PCT', 'AGE_25_34_PCT','FAMILY_HH_CHILD_LT18_PCT', 'VETERAN_POP_PCT', 'FAMILY_HH_TOTAL'] 
test_df[test_features].isnull().sum()

RACE_BLACK_NH_PCT                      0
INDIVIDUALS_NOT_IN_FAMILY_UNITS_PCT    0
AGE_25_34_PCT                          0
FAMILY_HH_CHILD_LT18_PCT               0
VETERAN_POP_PCT                        0
FAMILY_HH_TOTAL                        0
dtype: int64

In [11]:
prediction = best_model.predict(test_df[test_features])

pred_positive = []

for i in prediction:
    if i < 0:
        i = -i
    pred_positive.append(i)

sub['HOMELESS_RATE'] = pred_positive
# sub.to_csv("submission.csv", index=False)     
sub_cat = sub.copy()

In [12]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error

X = train_df[features].drop('HOMELESS_RATE', axis=1)
y = train_df['HOMELESS_RATE']

model_xgb = XGBRegressor()

param_grid = {
    'n_estimators': [100,150,200,250,300],
    'learning_rate': [0.01,0.05, 0.1,0.2],
    'max_depth': [3,4,5,6,7],
    'min_child_weight': [1, 2, 3, 4, 5],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0]
}

grid_search = GridSearchCV(estimator=model_xgb, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X, y)

best_params = grid_search.best_params_
best_model = grid_search.best_estimator_
print(best_params)
print(best_model)

best_model.fit(X, y)

y_pred = best_model.predict(X)

rmse = np.sqrt(mean_squared_error(y, y_pred))

print(rmse)



{'colsample_bytree': 0.6, 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 2, 'n_estimators': 150, 'subsample': 1.0}
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=4, max_leaves=None,
             min_child_weight=2, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=150, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)
0.0012142624735258546


In [13]:
# test_df[features] = test_df[features].fillna(0)
test_df = test_df[test_features]

predictions = best_model.predict(test_df)


sub['HOMELESS_RATE'] = predictions
# sub.to_csv("submission.csv", index=False)
sub_xgb = sub.copy()
sub_xgb

,ID,HOMELESS_RATE
0,AL_13,0.002941
1,LA_10,0.016611
2,SD_15,0.001545
3,SB_11,0.006848
4,SB_09,0.001860
5,OC_28,0.000331
6,AL_02,0.001769
7,RV_29,0.000987
8,SB_01,0.005546
9,SC_14,0.001861


In [14]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression

model_lr = LinearRegression()

X = train_df[features].drop('HOMELESS_RATE', axis=1)
y = train_df['HOMELESS_RATE']

param_grid = {
    'fit_intercept': [True, False],
    'copy_X': [True, False],
    'positive': [True, False],
    'n_jobs': [-1, None]
}

grid_search = GridSearchCV(estimator=model_lr, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X, y)

best_params = grid_search.best_params_
best_model = grid_search.best_estimator_
print(best_params)
print(best_model)

best_model.fit(X, y)

y_pred = best_model.predict(X)

rmse = np.sqrt(mean_squared_error(y, y_pred))

print(rmse)


{'copy_X': True, 'fit_intercept': False, 'n_jobs': -1, 'positive': True}
LinearRegression(fit_intercept=False, n_jobs=-1, positive=True)
0.006163615822418003


In [15]:
test_df = test_df[test_features]

predictions = best_model.predict(test_df)
predictions

array([0.00499182, 0.00755024, 0.00349325, 0.00340403, 0.00384857,
       0.00274601, 0.00407745, 0.00381998, 0.00568046, 0.00125144,
       0.00432409, 0.00423221, 0.00437539, 0.0044258 , 0.00572105,
       0.00475116, 0.00148025, 0.00409846, 0.00455799, 0.00264472,
       0.00425784, 0.00442261, 0.00493328, 0.00469379, 0.00406214,
       0.00338077, 0.00308438, 0.00481857, 0.00338622, 0.00562186,
       0.00508257, 0.00417077, 0.00521027, 0.00169108, 0.005693  ,
       0.00334702, 0.00402668, 0.00410695, 0.00456045, 0.004316  ,
       0.00665729, 0.00284123, 0.00539565, 0.00463972, 0.00352608,
       0.00563123, 0.00345185, 0.00336067, 0.0042907 , 0.0035784 ,
       0.00167906, 0.00538872, 0.00372883, 0.00523702, 0.00331399,
       0.00392878])

In [16]:
sub['HOMELESS_RATE'] = predictions
sub.to_csv("submission_lr.csv", index=False)
sub_lr = sub.copy()

In [17]:
y = (sub_cat['HOMELESS_RATE'] + sub_lr['HOMELESS_RATE'] + sub_xgb['HOMELESS_RATE'])/3
sub['HOMELESS_RATE'] = y
# sub.to_csv("submission.csv", index=False)
sub

,ID,HOMELESS_RATE
0,AL_13,0.003980
1,LA_10,0.012499
2,SD_15,0.002164
3,SB_11,0.005037
4,SB_09,0.002684
5,OC_28,0.001131
6,AL_02,0.002359
7,RV_29,0.002090
8,SB_01,0.004771
9,SC_14,0.001210


In [18]:
y = (sub_cat['HOMELESS_RATE']*99 + sub_xgb['HOMELESS_RATE'])/100
sub['HOMELESS_RATE'] = y
sub.to_csv("submission.csv", index=False)

In [19]:
y = sub_cat['HOMELESS_RATE']
sub['HOMELESS_RATE'] = y
# sub.to_csv("submission.csv", index=False)